# RealData LangGraph NL-to-Cypher Pipeline for PostgreSQL AGE

Production-style retrieval flow with explicit verification steps:

`question -> schema -> generate cypher -> verify -> execute -> repair if needed -> answer`

This notebook keeps data grounded in the graph. The model proposes Cypher, but Python validates safety and schema usage before execution.


In [ ]:
# Install only if your notebook environment does not already have these packages.
# ! pip install langgraph openai "psycopg[binary]" python-dotenv


In [ ]:
import json
import os
import re
from typing import Any, Dict, List, Optional, TypedDict

import psycopg
from dotenv import load_dotenv
from langgraph.graph import END, StateGraph
from openai import AzureOpenAI

load_dotenv(override=True)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

GRAPH_NAME = "realdata_knowledge_spine"

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint="https://ciaiciath2-foundry-dev.cognitiveservices.azure.com/",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

models = ["gpt-5.6-luna", "gpt-5.4-mini"]
NL2CYPHER_MODEL = models[0]
ANSWER_MODEL = models[0]
MAX_REPAIR_ATTEMPTS = 1


In [ ]:
def connect_postgres():
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


def normalize_agtype(value):
    if value is None:
        return None
    if isinstance(value, (list, dict, int, float, bool)):
        return value
    text = str(value).strip()
    try:
        return json.loads(text)
    except Exception:
        return text.strip('"')


In [ ]:
def get_age_node_schema(conn, graph_name):
    nodes = {}
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (n)
            RETURN DISTINCT labels(n), keys(n)
        $$) AS (labels agtype, properties agtype);
        """)
        rows = cursor.fetchall()

    for labels_value, properties_value in rows:
        labels = normalize_agtype(labels_value)
        properties = normalize_agtype(properties_value)
        if not isinstance(labels, list):
            labels = [labels]
        if not isinstance(properties, list):
            properties = [properties]
        for label in labels:
            nodes.setdefault(str(label), set()).update(str(prop) for prop in properties)
    return {label: sorted(properties) for label, properties in nodes.items()}


def get_age_relationship_schema(conn, graph_name):
    relationships = []
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (a)-[r]->(b)
            RETURN DISTINCT labels(a), type(r), labels(b)
        $$) AS (source_labels agtype, relationship agtype, target_labels agtype);
        """)
        rows = cursor.fetchall()

    for source_value, relationship_value, target_value in rows:
        relationships.append({
            "source": normalize_agtype(source_value),
            "relationship": normalize_agtype(relationship_value),
            "target": normalize_agtype(target_value),
        })
    return relationships


def get_age_graph_schema(conn, graph_name):
    return {
        "nodes": get_age_node_schema(conn, graph_name),
        "relationships": get_age_relationship_schema(conn, graph_name),
    }


def build_schema_text(schema):
    lines = ["NODE LABELS AND PROPERTIES"]
    for label, properties in schema["nodes"].items():
        lines.append(f"\nNode: {label}")
        lines.append("Properties: " + ", ".join(properties))

    lines.append("\nRELATIONSHIPS")
    for relation in schema["relationships"]:
        source = relation["source"]
        target = relation["target"]
        if isinstance(source, list):
            source = ", ".join(source)
        if isinstance(target, list):
            target = ", ".join(target)
        lines.append(f"({source})-[:{relation['relationship']}]->({target})")
    return "\n".join(lines)


In [ ]:
FORBIDDEN_CYPHER = [
    "CREATE",
    "MERGE",
    "DELETE",
    "DETACH",
    "SET",
    "REMOVE",
    "DROP",
    "LOAD CSV",
    "FOREACH",
    "CALL",
]

ALLOWED_START = ("MATCH", "OPTIONAL MATCH", "WITH", "UNWIND")


def validate_column_name(name):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid column name: {name}")
    return name


def strip_string_literals(cypher):
    result = []
    in_quote = None
    escaped = False
    for char in cypher:
        if in_quote:
            if escaped:
                escaped = False
                continue
            if char == "\\":
                escaped = True
                continue
            if char == in_quote:
                in_quote = None
                result.append("''")
            continue
        if char in {"'", '"'}:
            in_quote = char
            continue
        result.append(char)
    return "".join(result)


def validate_read_only_cypher(cypher):
    normalized = strip_string_literals(cypher).upper().strip()
    for keyword in FORBIDDEN_CYPHER:
        if re.search(rf"\b{re.escape(keyword)}\b", normalized):
            raise ValueError(f"Unsafe Cypher detected: {keyword}")
    if not normalized.startswith(ALLOWED_START):
        raise ValueError("Cypher must start with a read-only clause")
    if " RETURN " not in f" {normalized} ":
        raise ValueError("Cypher must include RETURN")
    return True


def extract_schema_tokens(cypher):
    labels = set(re.findall(r":([A-Za-z_][A-Za-z0-9_]*)", cypher))
    relationships = set(re.findall(r"\[:([A-Za-z_][A-Za-z0-9_]*)", cypher))
    properties = set(re.findall(r"\.([A-Za-z_][A-Za-z0-9_]*)", cypher))
    return labels, relationships, properties


def validate_schema_usage(cypher, graph_schema):
    labels, relationships, properties = extract_schema_tokens(cypher)
    known_labels = set(graph_schema["nodes"])
    known_relationships = {str(rel["relationship"]) for rel in graph_schema["relationships"]}
    known_properties = {prop for props in graph_schema["nodes"].values() for prop in props}

    unknown_labels = sorted(labels - known_labels)
    unknown_relationships = sorted(relationships - known_relationships)
    unknown_properties = sorted(properties - known_properties)

    errors = []
    if unknown_labels:
        errors.append(f"Unknown labels: {unknown_labels}")
    if unknown_relationships:
        errors.append(f"Unknown relationships: {unknown_relationships}")
    if unknown_properties:
        errors.append(f"Unknown properties: {unknown_properties}")
    if errors:
        raise ValueError("; ".join(errors))
    return True


def validate_query_spec(query_spec, graph_schema):
    cypher = query_spec["cypher"].strip()
    columns = [validate_column_name(column) for column in query_spec["columns"]]
    validate_read_only_cypher(cypher)
    validate_schema_usage(cypher, graph_schema)
    return {"cypher": cypher, "columns": columns}


In [ ]:
def llm_json(messages, model):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)


def generate_query_with_llm(question, schema_text, previous_error=None, previous_cypher=None):
    repair_context = ""
    if previous_error:
        repair_context = f"""
Previous Cypher failed verification or execution.
Previous Cypher:
{previous_cypher}

Error:
{previous_error}

Generate a corrected query.
"""

    messages = [
        {
            "role": "system",
            "content": """
You generate read-only Apache AGE Cypher queries.
Return JSON only with this exact shape:
{"cypher": "MATCH ... RETURN ...", "columns": ["column_1"]}

Rules:
- Use only the supplied schema.
- Never invent labels, properties, or relationships.
- Do not include the PostgreSQL SELECT FROM cypher wrapper.
- Every RETURN expression must have an explicit alias.
- Prefer scalar properties instead of full vertices or edges.
- Read-only Cypher only.
""".strip(),
        },
        {
            "role": "user",
            "content": f"GRAPH SCHEMA:\n{schema_text}\n\nQUESTION:\n{question}\n\n{repair_context}",
        },
    ]
    return llm_json(messages, NL2CYPHER_MODEL)


In [ ]:
def execute_age_query(conn, graph_name, query_spec):
    cypher = query_spec["cypher"]
    columns = [validate_column_name(column) for column in query_spec["columns"]]
    column_definition = ", ".join(f"{column} agtype" for column in columns)
    sql = f"""
    SELECT *
    FROM cypher('{graph_name}', $$
        {cypher}
    $$) AS ({column_definition});
    """

    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(sql)
        rows = cursor.fetchall()

    return [
        {column: normalize_agtype(value) for column, value in zip(columns, row)}
        for row in rows
    ]


def generate_answer(question, query_result):
    messages = [
        {
            "role": "system",
            "content": "Answer using only the supplied graph query result. If empty, say no matching data was found. Be concise.",
        },
        {
            "role": "user",
            "content": f"QUESTION:\n{question}\n\nQUERY RESULT:\n{json.dumps(query_result, indent=2, default=str)}",
        },
    ]
    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=messages,
    )
    return response.choices[0].message.content.strip()


In [ ]:
class GraphQAState(TypedDict, total=False):
    question: str
    graph_name: str
    schema: Dict[str, Any]
    schema_text: str
    query_spec: Dict[str, Any]
    query_result: List[Dict[str, Any]]
    answer: str
    error: Optional[str]
    repair_attempts: int
    verified: bool
    executed: bool


def load_schema_node(state: GraphQAState) -> GraphQAState:
    conn = connect_postgres()
    try:
        schema = get_age_graph_schema(conn, state["graph_name"])
    finally:
        conn.close()
    return {**state, "schema": schema, "schema_text": build_schema_text(schema), "error": None}


def generate_cypher_node(state: GraphQAState) -> GraphQAState:
    query_spec = generate_query_with_llm(
        question=state["question"],
        schema_text=state["schema_text"],
        previous_error=state.get("error"),
        previous_cypher=(state.get("query_spec") or {}).get("cypher"),
    )
    return {**state, "query_spec": query_spec, "verified": False, "executed": False, "error": None}


def verify_cypher_node(state: GraphQAState) -> GraphQAState:
    try:
        query_spec = validate_query_spec(state["query_spec"], state["schema"])
        return {**state, "query_spec": query_spec, "verified": True, "error": None}
    except Exception as exc:
        return {**state, "verified": False, "error": str(exc)}


def execute_query_node(state: GraphQAState) -> GraphQAState:
    try:
        conn = connect_postgres()
        try:
            query_result = execute_age_query(conn, state["graph_name"], state["query_spec"])
        finally:
            conn.close()
        return {**state, "query_result": query_result, "executed": True, "error": None}
    except Exception as exc:
        return {**state, "executed": False, "error": str(exc)}


def repair_or_fail_node(state: GraphQAState) -> GraphQAState:
    attempts = state.get("repair_attempts", 0) + 1
    return {**state, "repair_attempts": attempts}


def answer_node(state: GraphQAState) -> GraphQAState:
    answer = generate_answer(state["question"], state.get("query_result", []))
    return {**state, "answer": answer}


In [ ]:
def route_after_verify(state: GraphQAState):
    if state.get("verified"):
        return "execute_query"
    if state.get("repair_attempts", 0) < MAX_REPAIR_ATTEMPTS:
        return "repair"
    return "fail"


def route_after_execute(state: GraphQAState):
    if state.get("executed"):
        return "answer"
    if state.get("repair_attempts", 0) < MAX_REPAIR_ATTEMPTS:
        return "repair"
    return "fail"


def fail_node(state: GraphQAState) -> GraphQAState:
    error = state.get("error") or "Unknown pipeline failure"
    return {**state, "answer": f"Unable to produce a verified graph answer. Error: {error}"}


workflow = StateGraph(GraphQAState)
workflow.add_node("load_schema", load_schema_node)
workflow.add_node("generate_cypher", generate_cypher_node)
workflow.add_node("verify_cypher", verify_cypher_node)
workflow.add_node("execute_query", execute_query_node)
workflow.add_node("repair", repair_or_fail_node)
workflow.add_node("answer", answer_node)
workflow.add_node("fail", fail_node)

workflow.set_entry_point("load_schema")
workflow.add_edge("load_schema", "generate_cypher")
workflow.add_edge("generate_cypher", "verify_cypher")
workflow.add_conditional_edges("verify_cypher", route_after_verify)
workflow.add_conditional_edges("execute_query", route_after_execute)
workflow.add_edge("repair", "generate_cypher")
workflow.add_edge("answer", END)
workflow.add_edge("fail", END)

graph_app = workflow.compile()
print("LangGraph retrieval workflow compiled")


In [ ]:
def ask_age_graph_langgraph(question, show_cypher=True, show_raw_result=False, show_state=False):
    final_state = graph_app.invoke({
        "question": question,
        "graph_name": GRAPH_NAME,
        "repair_attempts": 0,
    })

    if show_cypher and final_state.get("query_spec"):
        print("Generated Cypher:\n")
        print(final_state["query_spec"]["cypher"])
    if show_raw_result:
        print("\nRaw AGE Result:\n")
        print(json.dumps(final_state.get("query_result", []), indent=2, default=str))
    if show_state:
        debug_state = dict(final_state)
        if "schema_text" in debug_state:
            debug_state["schema_text"] = debug_state["schema_text"][:2000] + "..."
        print("\nFinal State:\n")
        print(json.dumps(debug_state, indent=2, default=str))

    return final_state.get("answer")


In [ ]:
# Example after real_tranformation.ipynb has loaded the graph.
# answer = ask_age_graph_langgraph(
#     "List the available dataset entities and their domains",
#     show_raw_result=True,
#     show_state=True,
# )
# print(answer)
